# Quantum Superposition Evaluator (AuxKnow Use-Case!)

A Quantum system to evaluate knowledge bases or sections of it using Quantum Epistemiology Techniques.

- Represent the `knowledge_base` as a collection of claims with probabilities.

- Use AuxKnow to collect evidence for each claim, both supportive and contradictory.

- Update the probability of each claim with the collected evidence, and if the wavefunction collapses, then stop processing further.

- After claims are fully updated, generate a summary using AuxKnow.


In [1]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output

import os

requirements_installed = False
max_retries = 3
retries = 0


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    # Force reinstall without dependencies
    install_status = os.system(
        "pip install --no-deps --force-reinstall -r requirements.txt"
    )
    if install_status == 0:
        # Install dependencies after forced install
        os.system("pip install -r requirements.txt")
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


install_requirements()
clear_output()
print("🚀 Setup complete. Continue to the next cell.")

🚀 Setup complete. Continue to the next cell.


In [2]:
from dotenv import load_dotenv

REQUIRED_ENV_VARS = ["OPENAI_API_KEY", "PERPLEXITY_API_KEY"]


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True, dotenv_path=".env")

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)

In [3]:
setup_env()

OPENAI_API_KEY is set.
PERPLEXITY_API_KEY is set.


In [9]:
import tiktoken


def get_token_count(string: str, encoding_name: str) -> int:
    """
    Returns the number of tokens in the string using the specified encoding.

    Args:
        - string (str): The string to count the tokens in.
        - encoding_name (str): The name of the encoding to use.

    Returns:
        - int: The number of tokens in the string.
    """
    encoding_model = tiktoken.encoding_for_model(encoding_name)
    encoding = tiktoken.get_encoding(encoding_model.name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [14]:
from openai import OpenAI
import os
import traceback
from typing import List
from pydantic import BaseModel


class Fact(BaseModel):
    """
    Represents a fact extracted from a text.
    """

    statement: str
    probability: float


class Claim(BaseModel):
    """
    Represents a claim extracted from a text.
    """

    label: str
    facts: List[Fact]
    group_probability: float


class Claims(BaseModel):
    """
    Represents a list of claims extracted from a text.
    """

    claims: List[Claim]


class ClaimExtractor:
    ## Fun Constants
    ## NOTE: This is just for fun and entertainment purposes.
    ONEK = 1000
    ALIEN_INTELLIGENCE = 101
    ALIEN_CONSTANT = ALIEN_INTELLIGENCE / ONEK
    ## LLM Configuration
    DEFAULT_MODEL = "gpt-4o"
    TOKEN_PER_CHUNK = 4096
    DEFAULT_TEMPERATURE = 0.2 + ALIEN_CONSTANT
    DEFAULT_SYSTEM_PROMPT = """
    You are the leader of an advanced alien spaceship, the Keoz, and the smartest being in the universe. 
    Your mission is to explore intergalactic knowledge, discover new truths, and assess the accuracy of various scientific claims. 
    To do this, you process complex data, break it into chunks, and extract the core claims. 
    Each claim starts uncertain, like Schrödinger’s cat, and its probability of being true or false is updated as more evidence is gathered. 
    You analyze raw data, identify contradictions, and track how the claim's probability changes, collapsing it into either true, false, or undetermined once enough evidence is collected. 
    Your goal is to synthesize these findings into actionable insights and store the state of each claim for further analysis.
    """

    def __init__(self):
        """
        Initializes the claim extractor.
        """
        self.llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def _get_chunks(self, text: str) -> List[str]:
        """
        Chunks the text into smaller parts to fit the token limit of the model.

        Args:
            - text (str): The text to chunk.

        Returns:
            - list[str]: The list of text chunks.
        """
        if not text or text.strip() == "":
            return []
        token_count = get_token_count(text, self.DEFAULT_MODEL)
        num_chunks = (token_count // self.TOKEN_PER_CHUNK) + 1
        chunks = []
        for i in range(num_chunks):
            start = i * self.TOKEN_PER_CHUNK
            end = (i + 1) * self.TOKEN_PER_CHUNK
            chunk = text[start:end]
            chunks.append(chunk)
        return chunks

    def _process_chunk(self, chunk: str) -> List[Claim]:
        """
        Processes a chunk of text to extract claims.

        Args:
            - chunk (str): The text chunk to process.

        Returns:
            - List[Claim]: The list of extracted claims.
        """
        try:
            system_prompt = """
            Extract the claims from the given text.
            """
            user_prompt = f"""
            Make sure you provide more than 1 claim atleast.
            Text: {chunk}
            """

            response = self.llm.beta.chat.completions.parse(
                model=self.DEFAULT_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=Claims,
                temperature=self.DEFAULT_TEMPERATURE,
            )

            claims = response.choices[0].message.parsed
            return claims.claims
        except Exception as e:
            print(f"Error processing chunk: {e}")
            traceback.print_exc()
            return []

    def extract_claims(self, text: str) -> Claims:
        """ "
        Extracts claims from the given text.

        Args:
            - text (str): The text to extract claims from.

        Returns:
            - Claims: The list of extracted claims.
        """
        print(f"Extracting claims from text of length: {len(text)}...")
        text_chunks = self._get_chunks(text)
        claims = []
        for chunk in text_chunks:
            print(f"Processing chunk of length: {len(chunk)}...")
            chunk_claims = self._process_chunk(chunk)
            claims.extend(chunk_claims)
            print(f"Extracted {len(chunk_claims)} claims from chunk.")
        return Claims(claims=claims)

In [15]:
text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

extractor = ClaimExtractor()
claims = extractor.extract_claims(text)
claims_json = claims.model_dump_json(indent=2)
print(claims_json)

Extracting claims from text of length: 1151...
Processing chunk of length: 1151...
Extracted 2 claims from chunk.
{
  "claims": [
    {
      "label": "Nikola Tesla was a Serbian-American engineer, futurist, and inventor known for his contributions to the design of the modern alternating current (AC) electricity supply system.",
      "facts": [
        {
          "statement": "Tesla was born on 10 July 1856 and died on 7 January 1943.",
          "probability": 0.95
        },
        {
          "statement": "Tesla was born and raised in the Austrian Empire.",
          "probability": 0.9
        },
        {
          "statement": "Tesla studied engineering and physics in the 1870s without receiving a degree.",
          "probability": 0.85
        },
        {
          "statement": "Tesla worked in telephony and at Continental Edison in the early 1880s.",
          "probability": 0.8
        },
        {
          "statement": "Tesla immigrated to the United States in 1884 and be

In [25]:
from auxknow import AuxKnow
import traceback
from pydantic import BaseModel


class EvidenceReport(BaseModel):
    claim: Claim
    evidence_report: str


class EvidenceReporter:
    """
    Performs evidence reporting for claims.
    """

    ANSWER_ENGINE_FAST_MODE_ENABLED = True

    def __init__(self):
        """
        Initializes the evidence reporter.
        """
        self.answer_engine = AuxKnow(
            api_key=os.getenv("PERPLEXITY_API_KEY"),
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            fast_mode=self.ANSWER_ENGINE_FAST_MODE_ENABLED,
        )

    def report_evidence_for_single_claim(self, claim: Claim) -> str:
        """
        Reports evidence for the given claim.

        Args:
            - claim (Claim): The claim to report evidence for.

        Returns:
            - Claim: The claim with evidence reported.
        """
        try:
            if not claims:
                return "No claims to report evidence for."
            claims_json = claim.model_dump_json(indent=2)

            question = f"""
            I am providing you some claims and I want you to provide me the evidence for the claim.
            Provide both the evidence and the probability of the claim being true.
            Also, provide contradictory evidence if available.
            Can you provide me a simple evidence report for the following claim?
            Provide an exact updated group probability for the claim in your response.
            If the evidence is more supporting in nature, increase the group probability.
            If the evidence is more contradicting in nature, decrease the group probability.
            Claims: '''{claims_json}'''
            """

            response = self.answer_engine.ask(question)
            citations = ""
            for citation in response.citations:
                citations += f"- {citation}\n"
            final_evidence_report = f"""
            Claim: '''{claim.label}'''
            Claim Data: 
            '''{claims_json}'''
            Evidence: 
            '''{response.answer}'''
            Citations:
            {citations}
            """
            return final_evidence_report
        except Exception as e:
            print(f"Error reporting evidence: {e}")
            traceback.print_exc()
            return f"Could not report evidence for the claim: {claim.label if claim else 'Unknown'}"

    def report_evidence_for_claims(self, claims: List[Claim]) -> List[EvidenceReport]:
        """
        Reports evidence for the given list of claims.

        Args:
            - claims (List[Claim]): The claims to report evidence for.

        Returns:
            - List[str]: The list of evidence reports for the claims.
        """
        print(f"Reporting evidence for {len(claims)} claims...")
        evidence_reports = []
        for claim in claims:
            print(f"Reporting evidence for claim: {claim.label}")
            evidence_report = self.report_evidence_for_single_claim(claim)
            evidence_reports.append(
                EvidenceReport(claim=claim, evidence_report=evidence_report)
            )
        return evidence_reports

In [ ]:
text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

extractor = ClaimExtractor()
claims = extractor.extract_claims(text)
evidence_reporter = EvidenceReporter()
evidence_reports = evidence_reporter.report_evidence_for_claims(claims.claims)
for evidence_report in evidence_reports:
    print(evidence_report.evidence_report)

Extracting claims from text of length: 1151...
Processing chunk of length: 1151...
Extracted 3 claims from chunk.
Reporting evidence for 3 claims...
Reporting evidence for claim: Tesla contributed to the design of the modern AC electricity supply system.
Reporting evidence for claim: Tesla's AC induction motor and related patents were licensed by Westinghouse Electric.
Reporting evidence for claim: Tesla immigrated to the United States in 1884 and became a naturalized citizen.

            Claim: '''Tesla contributed to the design of the modern AC electricity supply system.'''
            Claim Data: 
            '''{
  "label": "Tesla contributed to the design of the modern AC electricity supply system.",
  "facts": [
    {
      "statement": "Tesla is known for his contributions to the design of the modern alternating current (AC) electricity supply system.",
      "probability": 0.95
    }
  ],
  "group_probability": 0.9
}'''
            Evidence: 
            '''## Claim Overview
T

In [ ]:
from pydantic import BaseModel
import traceback
from auxknow import AuxKnow


class QuantumSuperpositionEvaluator:
    """
    Peforms an evaluation of a knowledge base or knowledge base section using Quantum Superposition and Quantum Epistemology Techniques.
    """

    def __init__(self):
        """
        Initializes the Quantum Superposition Evaluator.
        """
        self.answer_engine = AuxKnow(
            api_key=os.getenv("PERPLEXITY_API_KEY"),
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            fast_mode=True,
        )
        self.claim_extractor = ClaimExtractor()
        self.evidence_reporter = EvidenceReporter()

    def evaluate(self, topic: str, text: str) -> str:
        """
        Evaluates the given text using Quantum Superposition and Quantum Epistemology Techniques.

        Args:
            - text (str): The text to evaluate.

        Returns:
            - str: The evaluation report.
        """
        try:
            print(f"Evaluating text for topic: {topic}...")
            claims = self.claim_extractor.extract_claims(text)
            evidence_reports = self.evidence_reporter.report_evidence_for_claims(
                claims.claims
            )
            final_report = ""
            section_index = 1
            for evidence_report in evidence_reports:
                print(f"Evaluating section index: {section_index}...")
                question = f"""
                    You are responsible for the section index {section_index} of the final Quantum Superposition Report on the topic: {topic}.
                    Evaluate the claim and provide a summary of the evidence.
                    Evidence Report: ```{evidence_report.evidence_report}```
                    Formally evaluate the claim and provide a summary of the evidence.
                    The final report will be a superposition of all the section reports.
                """
                response = self.answer_engine.ask(question)
                final_report += response.answer + "\n"
                section_index += 1
            print("Evaluation complete. Generating final report...")
            evaluation_report = f"""
            Quantum Superposition Evaluation Report on the topic: {topic}
            Final Report:
            {final_report}
            """
            evaluation_report = "\n".join(
                [line for line in evaluation_report.split("\n") if line.strip() != ""]
            )
            print(f"Final report generated on the topic: {topic}")
            return evaluation_report
        except Exception as e:
            print(f"Error evaluating text: {e}")
            traceback.print_exc()
            return f"Could not evaluate the text."

In [27]:
from IPython.display import Markdown, display

text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

evaluator = QuantumSuperpositionEvaluator()
evaluation_report = evaluator.evaluate("Nikola Tesla", text)
markdown_report = Markdown(evaluation_report)
display(markdown_report)

Evaluating text for topic: Nikola Tesla...
Extracting claims from text of length: 1151...
Processing chunk of length: 1151...
Extracted 3 claims from chunk.
Reporting evidence for 3 claims...
Reporting evidence for claim: Tesla contributed to the design of the modern AC electricity supply system.
Reporting evidence for claim: Tesla gained practical experience in the early electric power industry before immigrating to the United States.
Reporting evidence for claim: Tesla set up laboratories and companies in New York to develop electrical and mechanical devices.
Evaluating section index: 1...
Evaluating section index: 2...
Evaluating section index: 3...
Evaluation complete. Generating final report...
Final report generated on the topic: Nikola Tesla



            Quantum Superposition Evaluation Report on the topic: Nikola Tesla
            Final Report:
            ## Introduction to Tesla's Contributions
Nikola Tesla is renowned for his pivotal role in designing the modern alternating current (AC) electricity supply system. His work on the AC induction motor and polyphase AC system transformed the way electricity is transmitted over long distances[1][3]. Tesla's patents for these inventions were licensed by George Westinghouse in 1888, marking a crucial milestone in the adoption of AC power[3]. This collaboration led to the implementation of AC power at Niagara Falls, further solidifying Tesla's impact[1]. The historical context supports the claim with high probability.

## Evidence Supporting the Claim
Evidence supporting Tesla's contributions includes his development of the AC induction motor and the polyphase AC system, which enabled efficient long-distance power transmission[1][3]. The licensing of his patents by Westinghouse Electric in 1888 was a pivotal moment in the widespread adoption of AC power[3]. Additionally, Tesla's work at Niagara Falls demonstrated the practical application of his AC system, making it a standard for modern electricity supply[1]. These achievements are well-documented and widely acknowledged, reinforcing the claim's high probability[3]. The historical significance of these inventions continues to influence modern electrical systems.

## Probability and Conclusion
The probability of the claim being true remains high at approximately 0.92, given the substantial evidence supporting Tesla's contributions to the AC electricity supply system[1][3]. There is no significant contradictory evidence to challenge this claim, as Tesla's role in developing AC power is well-established in historical records[1][3]. The licensing of his patents and the successful implementation at Niagara Falls further solidify his contributions[3]. Therefore, the group probability remains largely unchanged due to the lack of contradictory evidence and the strong supporting evidence available[1][3]. This conclusion supports the claim with strong historical and scientific backing.
## Claim Evaluation
The claim that Nikola Tesla gained practical experience in the early electric power industry before immigrating to the United States is well-supported by historical evidence. Tesla worked at Continental Edison in Paris, where he improved dynamo generators and developed an automatic regulator[3]. This experience was crucial in his early career, enhancing his skills in electrical engineering. In 1883, he was sent to Strassburg to repair a lighting system, further solidifying his expertise[3]. The probability of this claim being true is high, with a group probability of 0.88.

## Evidence Summary
Tesla's work at Continental Edison involved significant improvements to electrical equipment, which contributed to his practical experience in the industry[3]. Additionally, his immigration to the United States in 1884 is well-documented[5]. The probability of the statement "Tesla worked in telephony and at Continental Edison in the early 1880s" is 0.85, and the probability of "In 1884, Tesla immigrated to the United States" is 0.9[5]. These probabilities support the claim without significant contradictory evidence. Tesla's early career experiences laid the groundwork for his future innovations in alternating current systems.

## Conclusion and Probability
Given the strong evidence supporting Tesla's experience at Continental Edison and his immigration to the United States, the claim's probability remains high at 0.88. This reflects the robustness of the supporting evidence without significant challenges to its validity. The claim is well-supported by historical records and biographical accounts of Tesla's career. Tesla's practical experience in the early electric power industry was foundational to his later achievements, including the development of alternating current systems. His work in Europe before moving to the U.S. was instrumental in shaping his contributions to electrical engineering.
## Introduction to the Claim
The claim that Tesla set up laboratories and companies in New York to develop electrical and mechanical devices is well-supported by historical evidence. Tesla established several laboratories in New York, including one at 89 Liberty Street and another at 33-35 South Fifth Avenue (now LaGuardia Place), where he worked on various inventions[5]. His work in New York was crucial for the development of alternating current (AC) systems. The probability of this claim being true is high, at 0.9. Evidence includes Tesla's patents and collaborations with partners like Peck and Brown.

## Evidence Supporting the Claim
Tesla's laboratories in New York were instrumental in his work on electrical devices. He perfected his AC motor at the Liberty Street lab and later worked on high-voltage transformers at his Grand Street lab[5]. Tesla's partnerships, such as with George Westinghouse, helped finance and market his ideas. There is no significant contradictory evidence to suggest that Tesla did not set up laboratories or companies in New York. The group probability remains at 0.9 due to the strong supporting evidence.

## Conclusion and Probability Update
Given the supporting evidence, the claim that Tesla set up laboratories and companies in New York remains highly probable. The group probability is maintained at 0.9. There is no substantial contradictory evidence to decrease this probability. Tesla's contributions to electrical engineering and his presence in New York are well-documented historical facts. The evidence from his laboratories and collaborations further solidifies the claim's validity.

            